---
title: "Spatialize Evaluation Module"
format:
  html:
    code-fold: false
    toc: true
  ipynb:
    code-fold: false
jupyter: python3
---

# Spatialize Evaluation Module

This notebook demonstrates the full functionality of `spatialize.evaluation`, which provides:

- **Metrics** — continuous (MAE, RMSE, R², bias, operational variants) and categorical (accuracy, precision, recall, F1)
- **Cross-validation** — leave-one-out and k-fold for any interpolation function
- **Benchmark utilities** — scipy and kriging wrappers, including auto-selection
- **Case studies** — `SyntheticScenario` (2D/3D, continuous/categorical) and `PrecipitationCaseStudy` (real 2.5D data)

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spatialize.gs.esi import esi_nongriddata, esi_griddata
from spatialize.gs.cat_esi import cat_esi_nongriddata
import spatialize.evaluation as ev
from spatialize.evaluation import (
    # Metrics — continuous
    error, absolute_error, bias,
    MAE, MSE, RMSE, R2,
    op_error, op_mae, op_rmse, op_mse,
    # Metrics — categorical
    accuracy, precision, recall, f1_score,
    # Cross-validation
    loo_validation, kfold_validation,
    # Benchmark
    scipy_prediction, auto_scipy_prediction,
    # Case studies
    SyntheticScenario, PrecipitationCaseStudy,
    # Utility
    SuppressOutput,
)

---

# Part 1 — Continuous Metrics

In [ ]:
true  = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
pred  = np.array([1.2, 1.8, 3.5, 3.9, 5.5])

print("Point-wise error  :", error(true, pred))
print("Absolute error    :", absolute_error(true, pred))
print()
print(f"Bias  : {bias(true, pred):.4f}  (positive → systematic over-prediction)")
print(f"MAE   : {MAE(true, pred):.4f}")
print(f"MSE   : {MSE(true, pred):.4f}")
print(f"RMSE  : {RMSE(true, pred):.4f}")
print(f"R²    : {R2(true, pred):.4f}")

## Operational (normalised) Metrics

Operational metrics divide by the dynamic range of the true values, making them comparable across datasets with different scales.

In [ ]:
print(f"OpMAE   : {op_mae(true, pred):.4f}")
print(f"OpRMSE  : {op_rmse(true, pred):.4f}")
print(f"OpMSE   : {op_mse(true, pred):.4f}")

## NaN Handling

Metric functions silently ignore positions where either array is NaN.

In [ ]:
true_nan = np.array([1.0, np.nan, 3.0, 4.0])
pred_nan = np.array([1.2,    2.0, 3.5, np.nan])

print(f"MAE with NaNs: {MAE(true_nan, pred_nan):.4f}  (only 2 valid pairs used)")

---

# Part 2 — Categorical Metrics

In [ ]:
true_cat = np.array(["A", "B", "A", "C", "B", "A"])
pred_cat = np.array(["A", "B", "C", "C", "A", "A"])

print(f"Accuracy  : {accuracy(true_cat, pred_cat):.4f}")
print(f"Precision : {precision(true_cat, pred_cat):.4f}  (weighted)")
print(f"Recall    : {recall(true_cat, pred_cat):.4f}  (weighted)")
print(f"F1 score  : {f1_score(true_cat, pred_cat):.4f}  (weighted)")

### Averaging strategies

In [ ]:
for avg in ('weighted', 'macro', 'micro'):
    print(f"  {avg:8s}  precision={precision(true_cat, pred_cat, average=avg):.3f}"
          f"  recall={recall(true_cat, pred_cat, average=avg):.3f}"
          f"  f1={f1_score(true_cat, pred_cat, average=avg):.3f}")

---

# Part 3 — Synthetic Case Studies

`SyntheticScenario` generates synthetic spatial data with known ground truth, enabling objective evaluation of interpolation methods.

## 3.1 2D Continuous Scenario (non-gridded)

In [ ]:
scenario_2d = SyntheticScenario(
    n_dims=2,
    extent=[0, 100, 0, 100],
    griddata=False,
    n_grid_points=60,
)

points, values, xi, reference = scenario_2d.simulate_scenario(
    kind='cubic', n_samples=200, seed=42
)

print(f"Sample points : {points.shape}")
print(f"Target grid   : {xi.shape}")
print(f"Reference     : {reference.shape}")

In [ ]:
fig, ax = scenario_2d.plot_2d_scenario(
    points, values, xi, reference,
    theme='publication',
    title='2D Continuous — Reference and Sample Points',
)
plt.show()

## 3.2 2D Gridded Continuous Scenario

In [ ]:
scenario_grid = SyntheticScenario(
    n_dims=2,
    extent=[0, 1, 0, 1],
    griddata=True,
    n_grid_points=60,
)

pts_g, vals_g, xi_g, ref_g = scenario_grid.simulate_scenario(
    kind='cubic', n_samples=300, seed=0
)

print(f"Grid xi shape : {xi_g.shape}  (2 × nx × ny)")
print(f"Reference     : {ref_g.shape}")

In [ ]:
fig, ax = scenario_grid.plot_2d_scenario(
    pts_g, vals_g, xi_g, ref_g,
    theme='publication',
    title='2D Gridded Continuous — Ground Truth',
)
plt.show()

## 3.3 2D Nominal Categorical Scenario

In [ ]:
scenario_cat = SyntheticScenario(
    categorical=True,
    n_grid_points=80,
)

pts_c, vals_c, xi_c, ref_c = scenario_cat.simulate_scenario(
    kind='nominal', n_samples=250, seed=7
)

print(f"Unique classes : {np.unique(ref_c)}")

In [ ]:
fig, ax = scenario_cat.plot_2d_scenario(
    pts_c, vals_c, xi_c, ref_c,
    title='Nominal Categorical Ground Truth',
)
plt.show()

## 3.4 2D Ordinal Categorical Scenario

In [ ]:
scenario_ord = SyntheticScenario(categorical=True, n_grid_points=80)

pts_o, vals_o, xi_o, ref_o = scenario_ord.simulate_scenario(
    kind='ordinal', n_samples=250, seed=7
)

ordinal_labels = list(SyntheticScenario.ORDINAL_ORDER)  # ['Low', 'Medium', 'High']
print(f"Ordinal classes : {ordinal_labels}")
print(f"Class counts    : { {k: int((ref_o == k).sum()) for k in ordinal_labels} }")

In [ ]:
fig, ax = scenario_ord.plot_2d_scenario(
    pts_o, vals_o, xi_o, ref_o,
    nonnum_order=ordinal_labels,
    title='Ordinal Categorical Ground Truth',
)
plt.show()

## 3.5 3D Continuous Scenario

In [ ]:
scenario_3d = SyntheticScenario(
    n_dims=3,
    extent=[0, 10, 0, 10, 0, 5],
    griddata=False,
    n_grid_points=(20, 20, 10),
)

pts_3, vals_3, xi_3, ref_3 = scenario_3d.simulate_scenario(
    kind='cubic', n_samples=400, seed=99
)

print(f"Sample points : {pts_3.shape}")
print(f"Target grid   : {xi_3.shape}  ({20*20*10} total locations)")

## 3.6 Custom Function Scenario

You can inject your own ground-truth function via `custom_func`:

In [ ]:
def gaussian_bump(x, y, cx=50, cy=50, sigma=15):
    return np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * sigma**2))

scenario_custom = SyntheticScenario(
    n_dims=2, extent=[0, 100, 0, 100], griddata=False, n_grid_points=60
)

pts_cu, vals_cu, xi_cu, ref_cu = scenario_custom.simulate_scenario(
    n_samples=200, seed=1,
    custom_func=gaussian_bump,
    custom_func_params={'cx': 50, 'cy': 50, 'sigma': 15},
)

fig, ax = scenario_custom.plot_2d_scenario(
    pts_cu, vals_cu, xi_cu, ref_cu,
    title='Custom Function — Gaussian Bump',
    theme='minimal',
)
plt.show()

---

# Part 4 — Cross-Validation

## 4.1 Leave-One-Out Validation

`loo_validation` evaluates any interpolation callable using LOO. The callable must have the signature `f(points, values, xi, **kwargs) -> array`.

In [ ]:
def esi_predict(pts, vals, xi_, **kw):
    return esi_nongriddata(pts, vals, xi_, **kw).estimation()

loo_results = loo_validation(
    points, values,
    esi_predict,
    metrics=[MAE, RMSE, R2],
    # ESI kwargs:
    local_interpolator='idw',
    n_partitions=100,
    alpha=0.8,
    exponent=2.0,
)

print("LOO results:")
for name, val in loo_results.items():
    print(f"  {name:6s}: {val:.4f}")

### LOO with predictions array

In [ ]:
loo_results2, loo_preds = loo_validation(
    points, values,
    esi_predict,
    metrics=[MAE, RMSE],
    return_predictions=True,
    local_interpolator='idw',
    n_partitions=100,
    alpha=0.8,
    exponent=2.0,
)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(values, loo_preds, s=10, alpha=0.6)
ax.plot([values.min(), values.max()], [values.min(), values.max()],
        'r--', lw=1, label='1:1 line')
ax.set_xlabel('Observed')
ax.set_ylabel('LOO Predicted')
ax.set_title(f"LOO Scatter  (MAE={loo_results2['MAE']:.4f}, RMSE={loo_results2['RMSE']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()

## 4.2 K-Fold Cross-Validation

In [ ]:
kfold_results = kfold_validation(
    points, values,
    esi_predict,
    k=5,
    metrics=[MAE, RMSE, R2],
    seed=42,
    local_interpolator='idw',
    n_partitions=100,
    alpha=0.8,
    exponent=2.0,
)

print("5-fold CV results:")
for name, val in kfold_results.items():
    print(f"  {name:6s}: {val:.4f}")

### K-fold with out-of-fold predictions

In [ ]:
kfold_results2, kfold_preds = kfold_validation(
    points, values,
    esi_predict,
    k=5, seed=42,
    metrics=[MAE, RMSE],
    return_predictions=True,
    local_interpolator='idw',
    n_partitions=100,
    alpha=0.8,
    exponent=2.0,
)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(values, kfold_preds, s=10, alpha=0.6, color='steelblue')
ax.plot([values.min(), values.max()], [values.min(), values.max()],
        'r--', lw=1, label='1:1 line')
ax.set_xlabel('Observed')
ax.set_ylabel('Out-of-Fold Predicted')
ax.set_title(f"5-Fold Scatter  (MAE={kfold_results2['MAE']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()

### CV for categorical ESI

In [ ]:
def cat_esi_predict(pts, vals, xi_, **kw):
    return cat_esi_nongriddata(pts, vals, xi_, **kw).estimation()

loo_cat = loo_validation(
    pts_c, vals_c,
    cat_esi_predict,
    metrics=[accuracy, f1_score],
    classifier='knn_pca',
    n_partitions=100,
    alpha=0.8,
)

print("LOO categorical results:")
for name, val in loo_cat.items():
    print(f"  {name:10s}: {val:.4f}")

---

# Part 5 — Benchmark Utilities

## 5.1 Scipy Prediction

In [ ]:
for method in ('nearest', 'linear', 'cubic'):
    est = scipy_prediction(points, values, xi, method=method)
    mae = MAE(reference, est)
    print(f"  scipy {method:8s}  MAE={mae:.4f}")

## 5.2 Auto Scipy (LOO-based Method Selection)

In [ ]:
best_est = auto_scipy_prediction(points, values, xi, metric='mae')
print(f"\nBest scipy MAE on reference: {MAE(reference, best_est):.4f}")

## 5.3 ESI vs Scipy Comparison

In [ ]:
esi_result = esi_nongriddata(
    points, values, xi,
    local_interpolator='idw',
    n_partitions=200,
    alpha=0.8,
    exponent=2.0,
)
esi_est = esi_result.estimation()

comparison = {
    'scipy-nearest': MAE(reference, scipy_prediction(points, values, xi, method='nearest')),
    'scipy-linear' : MAE(reference, scipy_prediction(points, values, xi, method='linear')),
    'scipy-cubic'  : MAE(reference, scipy_prediction(points, values, xi, method='cubic')),
    'ESI-IDW'      : MAE(reference, esi_est),
}

print("\nMethod comparison (MAE vs ground truth):")
for method, mae in sorted(comparison.items(), key=lambda x: x[1]):
    print(f"  {method:15s}: {mae:.4f}")

---

# Part 6 — Precipitation Case Study (Real 2.5D Data)

`PrecipitationCaseStudy` wraps the Maipo Basin precipitation dataset. Sample points have 3D coordinates (UTM East, UTM North, elevation) and the target is precipitation at unsampled locations.

## 6.1 Loading and Visualizing Data

In [ ]:
case_study = PrecipitationCaseStudy()

print(f"Dates         : {case_study.dates}")

In [ ]:
fig, axs = case_study.plot_input_data()
plt.show()

In [ ]:
fig, ax = case_study.plot_interpolation_locations()
plt.show()

## 6.2 Running ESI for All Dates

In [ ]:
points_pp, values_pp, xi_pp = case_study.model_inputs()

esi_results_df = case_study.locs.copy()
esi_results_df.columns = pd.MultiIndex.from_tuples(
    [(c, '') for c in esi_results_df.columns]
)

esi_params = {}

for date in case_study.dates:
    result = esi_nongriddata(
        points=points_pp[date],
        values=values_pp[date],
        xi=xi_pp,
        local_interpolator='idw',
        n_partitions=300,
        alpha=0.8,
        exponent=2.0,
        seed=42,
    )

    esi_results_df[(date, 'value')]     = result.estimation()
    esi_results_df[(date, 'precision')] = result.precision(ev.op_mae)

    esi_params[date] = {'alpha': 0.8, 'exponent': 2.0}

params_df = pd.DataFrame(esi_params).T
params_df.index.name = None
print(params_df)

## 6.3 Visualising ESI Results

In [ ]:
fig, axs = case_study.plot_esi_results(
    esi_results_df,
    parameters=params_df,
    local_interpolator='idw',
    precision_function='Operational MAE',
    fig_title='ESI-IDW — Precipitation Estimation',
)
plt.show()

## 6.4 Model Comparison

Compare ESI against scipy interpolation methods for a single date.

In [ ]:
date = case_study.dates[0]

locs_xy = case_study.locs[['X', 'Y']].values

# Build a comparison results frame expected by plot_model_comparison
comp_df = case_study.locs.copy()

for method in ('nearest', 'linear'):
    est = scipy_prediction(
        points_pp[date][:, :2],   # use only x,y for scipy
        values_pp[date],
        locs_xy,
        method=method,
    )
    comp_df[(method, date, 'value')] = est

# ESI already computed above
comp_df[('esi', date, 'value')] = esi_results_df[(date, 'value')].values

fig, subfigs, ax0, axs_cmp = case_study.plot_model_comparison(
    comp_df, date=date,
    interpolators=['nearest', 'linear', 'esi'],
    names=['Nearest Neighbor', 'Linear', 'ESI-IDW'],
)
plt.show()

## 6.5 Quantitative Evaluation with `loo_validation`

We evaluate each interpolation method on the sample data with LOO CV.

In [ ]:
date = case_study.dates[0]

def scipy_predict_2d(pts, vals, xi_, method='linear'):
    return scipy_prediction(pts[:, :2], vals, xi_[:, :2], method=method)

def esi_predict_3d(pts, vals, xi_, **kw):
    return esi_nongriddata(pts, vals, xi_, **kw).estimation()

methods = {
    'scipy-nearest': (scipy_predict_2d, {'method': 'nearest'}),
    'scipy-linear' : (scipy_predict_2d, {'method': 'linear'}),
    'ESI-IDW'      : (esi_predict_3d,   {
        'local_interpolator': 'idw',
        'n_partitions': 100,
        'alpha': 0.8,
        'exponent': 2.0,
    }),
}

print(f"LOO CV on {date}  ({len(values_pp[date])} samples)\n")
for name, (fn, kwargs) in methods.items():
    res = loo_validation(
        points_pp[date] if name == 'ESI-IDW' else points_pp[date],
        values_pp[date],
        fn,
        metrics=[MAE, RMSE, R2],
        **kwargs,
    )
    print(f"  {name:15s}  MAE={res['MAE']:.2f}  RMSE={res['RMSE']:.2f}  R²={res['R2']:.3f}")

---

# Part 7 — SuppressOutput Utility

`SuppressOutput` is a context manager that silences all stdout (useful when running many ESI calls in loops).

In [ ]:
print("Before context manager")

with SuppressOutput():
    print("This line is suppressed")
    result_silent = esi_nongriddata(
        points, values, xi[:10],
        local_interpolator='idw',
        n_partitions=50, alpha=0.8, exponent=2.0,
    )

print(f"After context manager — estimation at first location: {result_silent.estimation()[0]:.4f}")

---

# Summary

| Module component | Demonstrated in |
|---|---|
| `error`, `absolute_error`, `bias` | Part 1 |
| `MAE`, `MSE`, `RMSE`, `R2` | Part 1 |
| `op_mae`, `op_rmse`, `op_mse` | Part 1 |
| NaN-safe metric behaviour | Part 1 |
| `accuracy`, `precision`, `recall`, `f1_score` | Part 2 |
| `SyntheticScenario` — 2D continuous (gridded / non-gridded) | Part 3 |
| `SyntheticScenario` — nominal categorical | Part 3 |
| `SyntheticScenario` — ordinal categorical | Part 3 |
| `SyntheticScenario` — 3D continuous | Part 3 |
| `SyntheticScenario` — custom function | Part 3 |
| `loo_validation` — continuous | Part 4 |
| `loo_validation` — categorical | Part 4 |
| `kfold_validation` — continuous | Part 4 |
| `scipy_prediction` | Part 5 |
| `auto_scipy_prediction` | Part 5 |
| `PrecipitationCaseStudy` — data loading and plots | Part 6 |
| `PrecipitationCaseStudy.plot_esi_results` | Part 6 |
| `PrecipitationCaseStudy.plot_model_comparison` | Part 6 |
| `loo_validation` on real data | Part 6 |
| `SuppressOutput` | Part 7 |